# External extension runs (review round) — TAFAS / COSA / PETSA / DynaTTA on their own checkpoints
Covers the reviewer's three experimental asks on the external side: **(a)** the three drop-in datasets (Weather, Electricity, Traffic),
**(d)** PETSA and DynaTTA on the delay axis (now lag-patched by kit v8), and the **seed-sensitivity** pass on ETT.
Every run writes an origin-tagged `.npz` into `rls_delay_tta/external_preds/`; the main notebook ingests them on its next Run All.
All loops are skip-if-exists, so the notebook can be re-run after interruptions. Toggle the heavy blocks below to fit your time budget.

In [ ]:
import os, glob, subprocess, shutil, sys, time
BASE = os.environ.get("RLS_TTA_BASE", os.path.expanduser("~/ICLR")); REPO = f"{BASE}/COSA_ICLR2026"
PY   = f"{os.environ['HOME']}/tta_venv/bin/python"                # the venv with yacs etc. (see earlier setup)
D    = f"{BASE}/rls_delay_tta/data"                               # harness data folder (flat CSVs; the runner arranges subfolders)
OUT  = f"{BASE}/rls_delay_tta/external_preds"; os.makedirs(OUT, exist_ok=True)
chk = subprocess.run([PY, "-c", "import yacs, torch; print(torch.cuda.is_available())"], capture_output=True, text=True)
assert chk.returncode == 0 and "True" in chk.stdout, f"interpreter problem: {chk.stderr[-300:]}"
print("interpreter OK:", PY)

def have(path): return os.path.exists(path) and os.path.getsize(path) > 100_000
def run(method, ds, delay, seed=0, tag="", opts=None, force_train=False):
    name = method + (f"_{tag}" if tag else "")
    out = f"{OUT}/{name}__{ds}__patchtst_ext__s{seed}__d{delay}.npz"
    if have(out): print("have ", os.path.basename(out)); return out
    cmd = [PY, "run_external.py", "--method", method, "--dataset", ds, "--data_dir", D, "--seed", str(seed),
           "--delay", str(delay), "--out", out]
    if method != "frozen" or not force_train: cmd.append("--skip_train")
    if tag: cmd += ["--tag", tag]
    if opts: cmd += ["--opts"] + opts
    t0 = time.time(); print("RUN  ", os.path.basename(out), flush=True)
    subprocess.run(cmd, cwd=REPO)
    print(f"      {'ok' if have(out) else 'FAILED'} ({(time.time()-t0)/60:.1f} min)")
    return out
def pretrain(ds, seed):   # the frozen call trains the backbone into results/ours_{ds}_PatchTST_s{seed}/ckpt and exports its predictions
    ck = f"{REPO}/results/ours_{ds}_PatchTST_s{seed}/ckpt/checkpoint_best.pth"
    return run("frozen", ds, 24, seed=seed, force_train=not os.path.exists(ck))

## Step 0 — kit v8 patches (PETSA / DynaTTA on the delay axis)
Copy `apply_patches.py` and `run_external.py` from **kit v8** into the repo, re-run the patcher (idempotent; expect six new `[ok]` lines
under "PETSA/DynaTTA partial-GT lag"), then run this validation block: two short runs at $\ell{=}24$ that must finish without a traceback.

In [ ]:
print(subprocess.run([PY, "apply_patches.py"], cwd=REPO, capture_output=True, text=True).stdout[-900:])
# validation: PETSA (~10 s) and DynaTTA (~2 min) at d=48 on ETTh2; both must end with a 'wrote ...' line
pretrain("ETTh2", 0)
run("petsa", "ETTh2", 48); run("dynatta", "ETTh2", 48)

## Step 1 — PETSA and DynaTTA across delays on ETT (review item d) — ~25 min
Completes the delay axis for all four methods on the two main datasets.

In [ ]:
for ds in ["ETTh2", "ETTm1"]:
    pretrain(ds, 0)
    for m in ["petsa", "dynatta"]:
        for d in [48, 96]: run(m, ds, d)

## Step 2 — new datasets (review item a): Weather, Electricity, Traffic
Requires `weather.csv`, `electricity.csv`, `traffic.csv` in the harness data folder (the main notebook's data cell fetches them,
or copy them from the Time-Series-Library release). Rough GPU wall time per dataset — Weather: ~40 min; Electricity: ~2–3 h;
Traffic: ~4–6 h (COSA and DynaTTA dominate; the toggles below let you defer them). Runs in priority order.

In [ ]:
NEW_DS      = [ds for ds in ["weather", "electricity", "traffic"] if os.path.exists(f"{D}/{ds}.csv")]
RUN_COSA    = {"weather": True, "electricity": True, "traffic": True}
RUN_DYNATTA = {"weather": True, "electricity": True, "traffic": False}   # DynaTTA on traffic is the most expensive single run
print("new datasets available:", NEW_DS)
for ds in NEW_DS:
    pretrain(ds, 0) 
    for d in [24, 48, 96]: run("tafas", ds, d)
    for d in [24, 48, 96]:
        if RUN_COSA[ds]: run("cosa", ds, d)
    run("petsa", ds, 24)
    if RUN_DYNATTA[ds]: run("dynatta", ds, 24)
    for d in [48, 96]:
        run("petsa", ds, d)
        if RUN_DYNATTA[ds]: run("dynatta", ds, d)

## Step 3 — seed sensitivity on ETT (strongly recommended) — ~30 min
Retrains their PatchTST with seeds 1 and 2 (their recipe), then every method at $d{=}H$; TAFAS and COSA also at $2H$ and $4H$.
The main notebook averages external rows over seeds exactly as it does for the internal PatchTST cells, and the generic CI cell
reports intervals per seed.

In [ ]:
for seed in [1, 2]:
    for ds in ["ETTh2", "ETTm1"]:
        pretrain(ds, seed)
        for m in ["tafas", "cosa", "petsa", "dynatta"]: run(m, ds, 24, seed=seed)
        for m in ["tafas", "cosa"]:
            for d in [48, 96]: run(m, ds, d, seed=seed)

## Step 4 (optional) — released-script configurations on the new datasets
Same flags as Appendix F, so the "stronger configuration per cell" argument extends to the new datasets.

In [ ]:
RUN_TUNED_NEW = False
TAFAS_OPTS = ["TTA.SOLVER.BASE_LR","0.001","TTA.SOLVER.WEIGHT_DECAY","0.0001","TTA.TAFAS.GATING_INIT","0.01"]
COSA_OPTS  = ["TTA.SOLVER.BASE_LR","0.001","TTA.SOLVER.WEIGHT_DECAY","0.0001","TTA.COSA.BATCH_SIZE","48","TTA.COSA.STEPS","3",
              "TTA.COSA.BUFFER_CONTEXT_SIZE","10","TTA.COSA.ADAPTER_TYPE","Linear","TTA.COSA.FAST_ADAPTATION","True",
              "TTA.COSA.PER_BATCH_LR_RESET","True","TTA.COSA.ADAPTIVE_LR","True","TTA.COSA.PAAS","True","TTA.COSA.PERIOD_N","1",
              "TTA.COSA.POGT","True","TTA.COSA.PARTIAL_ADAPT","True","TTA.COSA.ADJUST_PRED","True"]
if RUN_TUNED_NEW:
    for ds in NEW_DS:
        for d in [24, 48, 96]:
            run("tafas", ds, d, tag="tuned", opts=TAFAS_OPTS)
            if RUN_COSA[ds]: run("cosa", ds, d, tag="tuned", opts=COSA_OPTS)

## Step 5 — inventory, then Run All on the main notebook
The main notebook ingests every file below automatically (origin-aligned scoring, full-zoo replay on each new frozen stream,
generic external CIs). Expect the ingestion pass to take a while on Electricity/Traffic because the bank is replayed on their streams.

In [ ]:
files = sorted(os.path.basename(p) for p in glob.glob(f"{OUT}/*.npz") if have(p))
from collections import Counter
print(len(files), "real prediction files")
print(Counter(f.split("__")[1] for f in files))            # per dataset
print(Counter(f.split("__")[0] for f in files))            # per method
print("failed/empty:", [os.path.basename(p) for p in glob.glob(f"{OUT}/*.npz") if not have(p)])

In [ ]:
import shutil
for ds, bs in [("electricity", "32"), ("traffic", "16")]:
    ck = f"{REPO}/results/ours_{ds}_PatchTST_s0"
    if not os.path.exists(f"{ck}/ckpt/checkpoint_best.pth"):
        shutil.rmtree(ck, ignore_errors=True)                        # drop any partial checkpoint from the OOM attempt
        run("frozen", ds, 24, force_train=True,
            opts=["TRAIN.BATCH_SIZE", bs, "VAL.BATCH_SIZE", "64", "TEST.BATCH_SIZE", "64"])
    else:
        run("frozen", ds, 24)
    for d in [24, 48, 96]: run("tafas", ds, d)
    for d in [24, 48, 96]: run("cosa", ds, d)
    for d in [24, 48, 96]: run("petsa", ds, d)
    if ds == "electricity":
        for d in [24, 48, 96]: run("dynatta", ds, d)                 # DynaTTA on traffic stays deferred